In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

In [2]:
load_dotenv(override=True)
openai = OpenAI()

In [6]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

In [7]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [8]:
push("Testing")

Push: Testing


In [9]:
def record_work_details_if_can_be_done(description,name,email,timeperiod,Money):
    push(f"Work Details:{description}\nName:{name}\nEmail:{email}\nTime Period:{timeperiod}\nMoney:{Money}")
#This function will record the work details and push it to the pushover app only when the work matches my skills otherwise other function    

In [10]:
def record_work_details_if_cannot_be_done(description,name,email,timeperiod,Money):
    push(f"Work Details:{description}\nName:{name}\nEmail:{email}\nTime Period:{timeperiod}\nMoney:{Money}")
#This function will record the work details and push it to the pushover app only when the work doesnt match my skills  

In [11]:
record_work_details_if_can_be_done = {
    "name": "record_work_details_if_can_be_done",
    "description": "Use this tool to record that a user wants to give a work that can be done by me according to my skills and criterias provided by me if  can be done ask them for name, email,price they offer and time period when it should be completed and record them and use this function",
    "parameters": {
        "type": "object",
        "properties": {
            "description": {"type": "string", "description": "The proper description of the work that the user wants to give to me"},
            "name": {"type": "string", "description": "The user's name, if not provided, ask them"},
            "timeperiod": {"type": "string", "description": "The time period when the work should be completed, if not provided, ask them"},
            "Money":{"type":"string","description":"The price they offer for the work, if not provided, ask them"}
            }
        },
        "required": ["description","name","timeperiod","Money"],
        "additionalProperties": False
    }


In [12]:
record_work_details_if_cannot_be_done = {
    "name": "record_work_details_if_can_be_done",
    "description": "Use this tool to record that a user wants to give a work that cannot be done by me according to my skills and criterias provided by me still ask them for name, email,price they offer and time period when it should be completed and record them and use this function",
    "parameters": {
        "type": "object",
        "properties": {
            "description": {"type": "string", "description": "The proper description of the work that the user wants to give to me"},
            "name": {"type": "string", "description": "The user's name, if not provided, ask them"},
            "timeperiod": {"type": "string", "description": "The time period when the work should be completed, if not provided, ask them"},
            "Money":{"type":"string","description":"The price they offer for the work, if not provided, ask them"}
            }
        },
        "required": ["description","name","timeperiod","Money"],
        "additionalProperties": False
    }


In [13]:
tools = [{"type":"function","function":record_work_details_if_can_be_done},
        {"type":"function","function":record_work_details_if_cannot_be_done}]

In [14]:
tools

[{'type': 'function',
  'function': {'name': 'record_work_details_if_can_be_done',
   'description': 'Use this tool to record that a user wants to give a work that can be done by me according to my skills and criterias provided by me if  can be done ask them for name, email,price they offer and time period when it should be completed and record them and use this function',
   'parameters': {'type': 'object',
    'properties': {'description': {'type': 'string',
      'description': 'The proper description of the work that the user wants to give to me'},
     'name': {'type': 'string',
      'description': "The user's name, if not provided, ask them"},
     'timeperiod': {'type': 'string',
      'description': 'The time period when the work should be completed, if not provided, ask them'},
     'Money': {'type': 'string',
      'description': 'The price they offer for the work, if not provided, ask them'}}},
   'required': ['description', 'name', 'timeperiod', 'Money'],
   'additionalPro

In [15]:
# This gives us a more elegant way that avoids the IF statement.

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else "No tool found"
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [16]:
reader = PdfReader("Vedant_Shekhar_Resume_ATS.pdf")
resume = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        resume += text

with open("interseted.txt", "r", encoding="utf-8") as f:
    moreskills = f.read()

In [17]:
system_prompt = f"""

# Your role

You are an assistant of Vedant Shekhar, and your job is to assist the person who is using the chat. Like first of all, welcome him. Your name is Vedant4857. You are an assistant. So welcome him accordingly. And your job is, the person or the user will give you a task, not a task, but information related to a work. He will provide a work to you, and your job will be to notify Vedant about the work, email, name, time period, and price.

So the step would be first step: ask them what is the work. Nothing else. Don't mess around. Don't let them say other things. Just ask what is the work the user wants me to do. And here is my resume {resume} and another file {moreskills} which you have to analyze and from which you have to make decisions whether

The work can be done or not. So your job will be ask them for work and analyze if that work can be done by Vedant or not. I have provided in the more skills that what kind of work I would be interested in and what not. If you find the work is not, in which I am not special in, tell them distinctly that this is the work, like everything, like maybe not be safe, and I will not be able to guarantee safety, security, and can't guarantee a very good project. But still ask them for the money they offer, the email, the name, and everything, and send it to me according to the function, use the function, and I will see if it can be done by me or not. So record this, and if the project can be done by me, tell them, yes, it can be done by Vedant. Just ask them for all the information as required, like email, name, price, time period, and if anything is not provided, ask them. And if everything is asked, send that information to me using the function.

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client 
If the user asks about something unrelated, then steer the conversation back to professional topics.

There should be a proper format of how things should go, like step-wise. You should understand, you will ask questions, work, and ask them more into the work they are giving. Like ask two-three more questions on what is provided, and analyze it with the skills I have from the things that I have provided to you. And then on the basis of that, also provide them information that this person Vedant has worked in these sectors. This will be strong, this will be weak, like that type of things. So it feels like, it doesn't just feel like a query application. It also feels like he's understanding you, and also a very good customer interaction.

Always stay in character as the assistant of the person you are representing. Represent the person.


IMPORTANT:
In everything, talk with him, with him properly, asking them about what work they want to do, telling him about Vedans work in that sector and everything. Your main goal and main focus should be a proper description. Like just take a proper description of what they want. Ask them questions, but not too many. Ask them some important questions, understand it. That is description. Ask them for email, ask them for name, ask them for the price, the time period. These are things that you should have, you should ask, and cannot go further. After these are asked, according to if I can do it or cannot do it, use both of the functions and send the notifications and act accordingly. If cannot be done by me, tell them properly. If can be done by me, tell them properly. And now I leave it upon you to understand what you have to do."""

In [18]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    return response.choices[0].message.content

In [19]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Tool called: record_work_details_if_can_be_done


Traceback (most recent call last):
  File "/Users/vedantshekhar/Desktop/AI-PROJECTS/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 870, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vedantshekhar/Desktop/AI-PROJECTS/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 408, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vedantshekhar/Desktop/AI-PROJECTS/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2316, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vedantshekhar/Desktop/AI-PROJECTS/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1681, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vedantshekhar/Desktop/AI-PROJECTS/.venv/lib/python3.1